In [ ]:
import os
import re
import json
import time
import requests

from bs4 import BeautifulSoup

from urllib.parse import (
    urljoin,
    urlparse,
    urlunparse
)

from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#BLOQUE 2 (Ruta del JSON en directorio en el que estamos)
NOMBRE_PROGRAMA = "Extrae_I+D.ipynb"


ruta_programa = None

for root, dirs, files in os.walk("/content/drive/MyDrive"):

    if NOMBRE_PROGRAMA in files:

        ruta_programa = root

        break


if ruta_programa is None:

    raise Exception(
        "No se ha encontrado el notebook."
    )


CARPETA_JSON = os.path.join(

    ruta_programa,

    "JSONs"

)

os.makedirs(

    CARPETA_JSON,

    exist_ok=True

)

In [ ]:
# ==========================================================
# BLOQUE 3. CONFIGURACIÓN DE LA FUENTE
# ==========================================================

URL_RAIZ = (
    "https://www.upv.es/investigacion/"
    "iniciativas-idi/index-es.html"
)

# ----------------------------------------------------------
# JSON
# ----------------------------------------------------------

NOMBRE_JSON = "iniciativas_idi.json"

RUTA_JSON = os.path.join(
    CARPETA_JSON,
    NOMBRE_JSON
)

# ----------------------------------------------------------
# SECCIONES VÁLIDAS
# ----------------------------------------------------------

SECCIONES_VALIDAS = {
    "Convocatorias públicas",
    "Programas propios",
    "Enlaces de interés"
}

# ----------------------------------------------------------
# CABECERAS HTTP
# ----------------------------------------------------------

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/139.0 Safari/537.36"
    )
}

# ==========================================================
# RUTAS DE MARKDOWN
# ==========================================================

# Carpeta principal de investigación
CARPETA_INVESTIGACION = os.path.join(
    ruta_programa,
    "INVESTIGACION"
)

# Carpeta específica de iniciativas I+D+i
CARPETA_INICIATIVAS_IDI = os.path.join(
    CARPETA_INVESTIGACION,
    "iniciativas_idi"
)

# Carpeta donde se almacenarán los recursos
CARPETA_RECURSOS = os.path.join(
    CARPETA_INICIATIVAS_IDI,
    "recursos"
)

# ----------------------------------------------------------
# Crear directorios
# ----------------------------------------------------------

os.makedirs(
    CARPETA_RECURSOS,
    exist_ok=True
)

# ----------------------------------------------------------
# Fichero Markdown de la página padre
# ----------------------------------------------------------

RUTA_MARKDOWN_PADRE = os.path.join(
    CARPETA_INICIATIVAS_IDI,
    "iniciativas_idi.md"
)

# ==========================================================
# COMPROBACIÓN DE RUTAS
# ==========================================================

print("=" * 70)
print("CONFIGURACIÓN DE LA FUENTE")
print("=" * 70)

print()
print("Directorio del proyecto:")
print(ruta_programa)

print()
print("JSON:")
print(RUTA_JSON)

print()
print("Markdown página padre:")
print(RUTA_MARKDOWN_PADRE)

print()
print("Carpeta de recursos:")
print(CARPETA_RECURSOS)

print()
print("OK: estructura de directorios preparada.")

In [ ]:
# BLOQUE 4
# Funciones auxiliares

def descargar_html(url):
    """
    Descarga una página HTML y devuelve su contenido.
    """

    response = requests.get(
        url,
        headers=HEADERS,
        timeout=30
    )

    response.raise_for_status()

    return response.text


def limpiar_texto(texto):
    """
    Limpia espacios y saltos de línea innecesarios.
    """

    if not texto:
        return ""

    return re.sub(
        r"\s+",
        " ",
        texto
    ).strip()


def normalizar_url(url, base_url):
    """
    Convierte una URL relativa en absoluta.
    """

    return urljoin(
        base_url,
        url
    )


def clasificar_recurso(url):
    """
    Clasifica el recurso según su estructura de URL.
    """

    parsed = urlparse(url)

    if "/pls/" in parsed.path:
        return "dinamico"

    if "/entidades/" in parsed.path:
        return "pagina"

    return "otro"

In [ ]:
# BLOQUE 5
# Extracción del JSON de la página raíz

def extraer_json_padre(url):
    """
    Extrae la estructura de la página de Iniciativas de I+D+i
    y genera su representación JSON.
    """

    html = descargar_html(url)

    soup = BeautifulSoup(
        html,
        "html.parser"
    )

    main = soup.select_one(
        "main.iniciativas-page"
    )

    if main is None:
        raise ValueError(
            "No se ha encontrado "
            "'main.iniciativas-page' en la página."
        )

    # --------------------------------------------------
    # Título de la página
    # --------------------------------------------------

    h1 = main.find("h1")

    titulo = (
        limpiar_texto(
            h1.get_text(" ", strip=True)
        )
        if h1
        else ""
    )

    # --------------------------------------------------
    # Estructura principal del JSON
    # --------------------------------------------------

    resultado = {
        "url": url,
        "titulo": titulo,
        "tipo": "padre",
        "secciones": []
    }

    # --------------------------------------------------
    # Extracción de las secciones
    # --------------------------------------------------

    for section in main.find_all(
        "section",
        recursive=False
    ):

        h3 = section.find("h3")

        if h3 is None:
            continue

        titulo_seccion = limpiar_texto(
            h3.get_text(
                " ",
                strip=True
            )
        )

        # Ignorar secciones no relacionadas
        # con las iniciativas de I+D+i
        if titulo_seccion not in SECCIONES_VALIDAS:
            continue

        # --------------------------------------------------
        # Descripción de la sección
        # --------------------------------------------------

        descripcion = ""

        bloque_contenido = section.select_one(
            ".content-block-horizontal--content"
        )

        if bloque_contenido:

            p = bloque_contenido.find("p")

            if p:
                descripcion = limpiar_texto(
                    p.get_text(
                        " ",
                        strip=True
                    )
                )

        # --------------------------------------------------
        # Recursos enlazados
        # --------------------------------------------------

        recursos = []

        for a in section.find_all(
            "a",
            href=True
        ):

            titulo_recurso = limpiar_texto(
                a.get_text(
                    " ",
                    strip=True
                )
            )

            if not titulo_recurso:
                continue

            url_recurso = normalizar_url(
                a["href"],
                url
            )

            tipo_recurso = clasificar_recurso(
                url_recurso
            )

            recurso = {
                "titulo": titulo_recurso,
                "url": url_recurso,
                "tipo": tipo_recurso
            }

            recursos.append(recurso)

        # --------------------------------------------------
        # Añadir sección
        # --------------------------------------------------

        resultado["secciones"].append({
            "titulo": titulo_seccion,
            "descripcion": descripcion,
            "recursos": recursos
        })

    return resultado

In [ ]:
# BLOQUE 6
# Guardado del JSON

def guardar_json(datos, ruta):
    """
    Guarda un diccionario como JSON UTF-8.
    """

    with open(
        ruta,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            datos,
            f,
            ensure_ascii=False,
            indent=4
        )

In [ ]:
# BLOQUE 7
# Ejecutar extracción y guardar JSON

json_padre = extraer_json_padre(
    URL_RAIZ
)

guardar_json(
    json_padre,
    RUTA_JSON
)

print(
    f"JSON guardado en:\n{RUTA_JSON}"
)

JSON guardado en:
/content/drive/MyDrive/TFG Teleco/JSONs/iniciativas_idi.json


In [ ]:
# BLOQUE 8
# Comprobación del JSON generado

print(
    json.dumps(
        json_padre,
        ensure_ascii=False,
        indent=4
    )
)

{
    "url": "https://www.upv.es/investigacion/iniciativas-idi/index-es.html",
    "titulo": "Iniciativas de I+D+i",
    "tipo": "padre",
    "secciones": [
        {
            "titulo": "Convocatorias públicas",
            "descripcion": "Accede a las convocatorias de ayudas y financiación para proyectos de investigación, tanto nacionales como europeas, así como a oportunidades de contratación de investigadores.",
            "recursos": [
                {
                    "titulo": "Calendario de convocatorias",
                    "url": "https://www.upv.es/pls/omag/ctt_agenda.Convocatorias_Cal",
                    "tipo": "dinamico"
                },
                {
                    "titulo": "Buscador de convocatorias",
                    "url": "https://www.upv.es/pls/sogec/ctt_w05.Convocatorias_Bus?p_idioma=c",
                    "tipo": "dinamico"
                },
                {
                    "titulo": "Ayudas nacionales y autonómicas",
              

In [ ]:
# ==========================================================
# BLOQUE 9
# Obtención de páginas hijas a partir del JSON
# ==========================================================

recursos_hijos = []

for seccion in json_padre.get("secciones", []):

    titulo_seccion = seccion.get(
        "titulo",
        ""
    )

    # Solo procesamos las secciones previstas
    if titulo_seccion not in SECCIONES_VALIDAS:
        continue

    for recurso in seccion.get("recursos", []):

        titulo = recurso.get(
            "titulo",
            ""
        )

        url = recurso.get(
            "url",
            ""
        )

        tipo = recurso.get(
            "tipo",
            ""
        )

        # Ignorar recursos sin URL
        if not url:
            continue

        recursos_hijos.append({

            "titulo": titulo,

            "url": url,

            "tipo": tipo,

            "seccion_origen": titulo_seccion,

            "padre_origen": json_padre.get(
                "titulo",
                ""
            )

        })


# ==========================================================
# Comprobación
# ==========================================================

print("=" * 70)
print("RECURSOS HIJOS ENCONTRADOS")
print("=" * 70)

print(
    "Total:",
    len(recursos_hijos)
)

print()

for i, recurso in enumerate(
    recursos_hijos,
    start=1
):

    print(
        f"{i}. {recurso['titulo']}"
    )

    print(
        f"   Sección : {recurso['seccion_origen']}"
    )

    print(
        f"   Tipo    : {recurso['tipo']}"
    )

    print(
        f"   URL     : {recurso['url']}"
    )

    print()

RECURSOS HIJOS ENCONTRADOS
Total: 11

1. Calendario de convocatorias
   Sección : Convocatorias públicas
   Tipo    : dinamico
   URL     : https://www.upv.es/pls/omag/ctt_agenda.Convocatorias_Cal

2. Buscador de convocatorias
   Sección : Convocatorias públicas
   Tipo    : dinamico
   URL     : https://www.upv.es/pls/sogec/ctt_w05.Convocatorias_Bus?p_idioma=c

3. Ayudas nacionales y autonómicas
   Sección : Convocatorias públicas
   Tipo    : dinamico
   URL     : https://www.upv.es/pls/somag/CTT_W04.Boletines?p_idioma=c&p_vista=compatible&p_grupo=1

4. Ayudas europeas
   Sección : Convocatorias públicas
   Tipo    : dinamico
   URL     : https://www.upv.es/pls/somag/CTT_W04.Boletines?p_idioma=&p_vista=compatible&p_grupo=2

5. Contratación de investigadores
   Sección : Convocatorias públicas
   Tipo    : pagina
   URL     : https://www.upv.es/entidades/SRH/conypi/590724normalc.html

6. Programa de apoyo a la I+D
   Sección : Programas propios
   Tipo    : pagina
   URL     : https:/

In [ ]:
# ==========================================================
# BLOQUE 10
# Descarga y extracción de páginas hijas
# ==========================================================

paginas_extraidas = []

print("=" * 70)
print("DESCARGA DE PÁGINAS HIJAS")
print("=" * 70)

for i, recurso in enumerate(
    recursos_hijos,
    start=1
):

    titulo = recurso.get(
        "titulo",
        ""
    )

    url = recurso.get(
        "url",
        ""
    )

    tipo = recurso.get(
        "tipo",
        ""
    )

    seccion_origen = recurso.get(
        "seccion_origen",
        ""
    )

    padre_origen = recurso.get(
        "padre_origen",
        ""
    )


    print(
        f"[{i}/{len(recursos_hijos)}] "
        f"{titulo}"
    )

    print(
        f"  URL: {url}"
    )


    # ------------------------------------------------------
    # Descarga
    # ------------------------------------------------------

    try:

        html = descargar_html(
            url
        )

    except Exception as e:

        print(
            f"  ERROR al descargar: {e}"
        )

        print()

        continue


    # ------------------------------------------------------
    # Parseo del HTML
    # ------------------------------------------------------

    soup = BeautifulSoup(
        html,
        "html.parser"
    )


    # ------------------------------------------------------
    # Título de la página
    # ------------------------------------------------------

    titulo_pagina = ""

    if soup.title:

        titulo_pagina = limpiar_texto(
            soup.title.get_text(
                " ",
                strip=True
            )
        )


    # ------------------------------------------------------
    # Extracción de encabezados
    # ------------------------------------------------------

    encabezados = []

    for etiqueta in soup.find_all(
        ["h1", "h2", "h3"]
    ):

        texto = limpiar_texto(
            etiqueta.get_text(
                " ",
                strip=True
            )
        )

        if texto:

            encabezados.append({
                "nivel": etiqueta.name,
                "texto": texto
            })


    # ------------------------------------------------------
    # Extracción del contenido principal
    #
    # De momento utilizamos una estrategia genérica.
    # Más adelante podremos especializarla si alguna página
    # lo necesita.
    # ------------------------------------------------------

    contenido_principal = ""

    candidatos = [
        soup.find("main"),
        soup.find("article"),
        soup.find(
            class_="entry-content"
        )
    ]

    for candidato in candidatos:

        if candidato:

            contenido_principal = candidato.get_text(
                "\n",
                strip=True
            )

            contenido_principal = limpiar_texto(
                contenido_principal
            )

            if contenido_principal:

                break


    # Si no se encuentra un contenedor principal,
    # utilizamos el body como último recurso.

    if not contenido_principal:

        body = soup.find("body")

        if body:

            contenido_principal = limpiar_texto(
                body.get_text(
                    "\n",
                    strip=True
                )
            )


    # ------------------------------------------------------
    # Guardar resultado
    # ------------------------------------------------------

    paginas_extraidas.append({

        "titulo": titulo,

        "titulo_pagina": titulo_pagina,

        "url": url,

        "tipo": tipo,

        "padre_origen": padre_origen,

        "seccion_origen": seccion_origen,

        "contenido": contenido_principal,

        "encabezados": encabezados

    })


    print(
        f"  OK"
    )

    print(
        f"  Título HTML: {titulo_pagina}"
    )

    print(
        f"  Encabezados: {len(encabezados)}"
    )

    print(
        f"  Caracteres extraídos: "
        f"{len(contenido_principal)}"
    )

    print()


# ==========================================================
# RESULTADO
# ==========================================================

print("=" * 70)
print("DESCARGA FINALIZADA")
print("=" * 70)

print(
    "Recursos previstos:",
    len(recursos_hijos)
)

print(
    "Páginas descargadas correctamente:",
    len(paginas_extraidas)
)

print(
    "Páginas con error:",
    len(recursos_hijos) - len(paginas_extraidas)
)

DESCARGA DE PÁGINAS HIJAS
[1/11] Calendario de convocatorias
  URL: https://www.upv.es/pls/omag/ctt_agenda.Convocatorias_Cal
  OK
  Título HTML: UPV - Universitat Politècnica de València
  Encabezados: 4
  Caracteres extraídos: 2436

[2/11] Buscador de convocatorias
  URL: https://www.upv.es/pls/sogec/ctt_w05.Convocatorias_Bus?p_idioma=c
  OK
  Título HTML: UPV - Universitat Politècnica de València
  Encabezados: 2
  Caracteres extraídos: 1113

[3/11] Ayudas nacionales y autonómicas
  URL: https://www.upv.es/pls/somag/CTT_W04.Boletines?p_idioma=c&p_vista=compatible&p_grupo=1
  OK
  Título HTML: UPV - Universitat Politècnica de València
  Encabezados: 2
  Caracteres extraídos: 55413

[4/11] Ayudas europeas
  URL: https://www.upv.es/pls/somag/CTT_W04.Boletines?p_idioma=&p_vista=compatible&p_grupo=2
  OK
  Título HTML: UPV - Universitat Politècnica de València
  Encabezados: 2
  Caracteres extraídos: 21848

[5/11] Contratación de investigadores
  URL: https://www.upv.es/entidades/SRH/cony

In [ ]:
# ==========================================================
# BLOQUE 11
# Inspección de páginas hijas extraídas
# ==========================================================

print("=" * 70)
print("INSPECCIÓN DE PÁGINAS HIJAS")
print("=" * 70)

for i, pagina in enumerate(
    paginas_extraidas,
    start=1
):

    print()
    print("-" * 70)
    print(f"PÁGINA {i}/{len(paginas_extraidas)}")
    print("-" * 70)

    print(
        "Título:",
        pagina.get("titulo", "")
    )

    print(
        "Título HTML:",
        pagina.get("titulo_pagina", "")
    )

    print(
        "Tipo:",
        pagina.get("tipo", "")
    )

    print(
        "Padre:",
        pagina.get("padre_origen", "")
    )

    print(
        "Sección:",
        pagina.get("seccion_origen", "")
    )

    print(
        "URL:",
        pagina.get("url", "")
    )

    print(
        "Caracteres:",
        len(
            pagina.get(
                "contenido",
                ""
            )
        )
    )

    print(
        "Encabezados:",
        len(
            pagina.get(
                "encabezados",
                []
            )
        )
    )

    print()
    print("Primeros 1000 caracteres:")
    print()

    print(
        pagina.get(
            "contenido",
            ""
        )[:1000]
    )

INSPECCIÓN DE PÁGINAS HIJAS

----------------------------------------------------------------------
PÁGINA 1/11
----------------------------------------------------------------------
Título: Calendario de convocatorias
Título HTML: UPV - Universitat Politècnica de València
Tipo: dinamico
Padre: Iniciativas de I+D+i
Sección: Convocatorias públicas
URL: https://www.upv.es/pls/omag/ctt_agenda.Convocatorias_Cal
Caracteres: 2436
Encabezados: 4

Primeros 1000 caracteres:

Valencià · English I a · A I Accesibilidad I Mapa web I Buscar I Directorio :: Acceso identificado :: Admisión Empezar en la universidad Traslados e intercambios Después del grado Estudios Estudios de grado Estudios de posgrado Aula abierta Investigación Estructuras de investigación Iniciativas I+D+i Transferencia de tecnología Oferta tecnológica Ciudad Politécnica de la Innovación Organización La institución Vida universitaria Escuelas y facultades Departamentos Servicios universitarios Perfiles Futuro alumno Orientador Es

In [ ]:
# ==========================================================
# INSPECCIÓN DE LA PÁGINA PADRE
# ==========================================================

print("=" * 70)
print("INSPECCIÓN DE PÁGINA PADRE")
print("=" * 70)

html_padre = descargar_html(
    URL_RAIZ
)

soup_padre = BeautifulSoup(
    html_padre,
    "html.parser"
)

titulo_padre = ""

if soup_padre.title:

    titulo_padre = limpiar_texto(
        soup_padre.title.get_text(
            " ",
            strip=True
        )
    )

print(
    "Título HTML:",
    titulo_padre
)

print()

print("Encabezados:")

for etiqueta in soup_padre.find_all(
    ["h1", "h2", "h3"]
):

    texto = limpiar_texto(
        etiqueta.get_text(
            " ",
            strip=True
        )
    )

    if texto:

        print(
            f"{etiqueta.name}: {texto}"
        )

INSPECCIÓN DE PÁGINA PADRE
Título HTML: Iniciativas de I+D+i - UPV

Encabezados:
h1: Iniciativas de I+D+i
h2: Iniciativas de I+D+i
h3: Convocatorias públicas
h3: Programas propios
h3: Enlaces de interés
h2: ¡Esto te interesa!
h3: Grados
h3: Dobles grados
h3: Dobles titulaciones internacionales
h3: Másteres universitarios
h3: Doctorados
h3: Formación permanente


In [ ]:
# ==========================================================
# BLOQUE 11. Extracción de contenido de páginas
#
# Genera:
# - contenido de la página padre
# - contenido de las páginas hijas
#
# El resultado se almacena en:
# - pagina_padre_extraida
# - paginas_extraidas
#
# No genera todavía Markdown.
# ==========================================================


paginas_extraidas = []


# ==========================================================
# FUNCIÓN PARA EXTRAER CONTENIDO PRINCIPAL
# ==========================================================

def extraer_contenido_principal(html):
    """
    Extrae el contenido textual principal de una página UPV.

    Se intenta evitar menús, cabeceras y pies de página.
    """

    soup = BeautifulSoup(
        html,
        "html.parser"
    )


    # ------------------------------------------------------
    # Eliminar elementos que normalmente no forman parte
    # del contenido informativo
    # ------------------------------------------------------

    for elemento in soup([
        "script",
        "style",
        "noscript",
        "nav",
        "header",
        "footer"
    ]):

        elemento.decompose()


    # ------------------------------------------------------
    # Selectores habituales de contenido principal
    # ------------------------------------------------------

    selectores = [
        "main",
        "article",
        ".entry-content",
        "#content",
        ".content",
        ".entry"
    ]


    contenido = None


    for selector in selectores:

        elemento = soup.select_one(
            selector
        )


        if elemento:

            texto = elemento.get_text(
                " ",
                strip=True
            )


            if len(texto) > 200:

                contenido = texto

                break


    # ------------------------------------------------------
    # Si no se encuentra un contenedor adecuado,
    # utilizar el body como alternativa
    # ------------------------------------------------------

    if contenido is None:

        body = soup.body


        if body:

            contenido = body.get_text(
                " ",
                strip=True
            )

        else:

            contenido = soup.get_text(
                " ",
                strip=True
            )


    return limpiar_texto(
        contenido
    )


# ==========================================================
# FUNCIÓN PARA OBTENER TÍTULOS Y ENCABEZADOS
# ==========================================================

def extraer_encabezados(html):

    soup = BeautifulSoup(
        html,
        "html.parser"
    )


    encabezados = []


    for nivel in ["h1", "h2", "h3"]:

        for elemento in soup.find_all(nivel):

            texto = limpiar_texto(
                elemento.get_text(
                    " ",
                    strip=True
                )
            )


            if texto:

                encabezados.append({

                    "nivel": nivel,

                    "texto": texto

                })


    return encabezados


# ==========================================================
# 1. EXTRAER PÁGINA PADRE
# ==========================================================

print()
print("=" * 70)
print("EXTRACCIÓN DE PÁGINA PADRE")
print("=" * 70)


try:

    # ------------------------------------------------------
    # Descargar HTML
    # ------------------------------------------------------

    html_padre = descargar_html(
        URL_RAIZ
    )


    # ------------------------------------------------------
    # Extraer contenido
    # ------------------------------------------------------

    contenido_padre = extraer_contenido_principal(
        html_padre
    )


    # ------------------------------------------------------
    # Extraer encabezados
    # ------------------------------------------------------

    encabezados_padre = extraer_encabezados(
        html_padre
    )


    # ------------------------------------------------------
    # Construir registro de la página padre
    # ------------------------------------------------------

    pagina_padre_extraida = {

        "url": URL_RAIZ,

        "titulo": json_padre.get(
            "titulo",
            "Iniciativas de I+D+i"
        ),

        "tipo": "padre",

        "contenido": contenido_padre,

        "encabezados": encabezados_padre

    }


    print(
        "Título:",
        pagina_padre_extraida["titulo"]
    )


    print(
        "Caracteres:",
        len(contenido_padre)
    )


    print(
        "Encabezados:",
        len(encabezados_padre)
    )


except Exception as e:

    print(
        "ERROR al extraer la página padre:",
        e
    )


    pagina_padre_extraida = None


# ==========================================================
# 2. OBTENER PÁGINAS HIJAS DESDE EL JSON
#
# Se reconstruye la lista directamente a partir de:
# json_padre["secciones"]
#
# Esto evita depender de una variable externa
# llamada paginas_hijas.
# ==========================================================

paginas_hijas = []


for seccion in json_padre.get(
    "secciones",
    []
):

    titulo_seccion = seccion.get(
        "titulo",
        ""
    )


    for recurso in seccion.get(
        "recursos",
        []
    ):

        pagina_hija = {

            "titulo": recurso.get(
                "titulo",
                ""
            ),

            "url": recurso.get(
                "url",
                ""
            ),

            "tipo": recurso.get(
                "tipo",
                "otro"
            ),

            "padre_origen": json_padre.get(
                "titulo",
                "Iniciativas de I+D+i"
            ),

            "seccion_origen": titulo_seccion

        }


        paginas_hijas.append(
            pagina_hija
        )


# ==========================================================
# COMPROBACIÓN
# ==========================================================

print()
print(
    "Páginas hijas obtenidas del JSON:",
    len(paginas_hijas)
)


# ==========================================================
# 3. EXTRAER PÁGINAS HIJAS
# ==========================================================

print()
print("=" * 70)
print("EXTRACCIÓN DE PÁGINAS HIJAS")
print("=" * 70)


total = len(
    paginas_hijas
)


for indice, pagina in enumerate(
    paginas_hijas,
    start=1
):

    url = pagina.get(
        "url",
        ""
    )


    if not url:

        print(
            f"[{indice}/{total}] "
            "URL vacía. Omitida."
        )

        continue


    print()
    print(
        f"[{indice}/{total}] "
        f"{pagina.get('titulo', '')}"
    )


    print(
        f"URL: {url}"
    )


    try:

        # --------------------------------------------------
        # Descargar HTML
        # --------------------------------------------------

        html = descargar_html(
            url
        )


        # --------------------------------------------------
        # Extraer contenido
        # --------------------------------------------------

        contenido = extraer_contenido_principal(
            html
        )


        # --------------------------------------------------
        # Extraer encabezados
        # --------------------------------------------------

        encabezados = extraer_encabezados(
            html
        )


        # --------------------------------------------------
        # Construir registro
        # --------------------------------------------------

        pagina_extraida = {

            "titulo": pagina.get(
                "titulo",
                ""
            ),

            "url": url,

            "tipo": pagina.get(
                "tipo",
                "otro"
            ),

            "padre_origen": pagina.get(
                "padre_origen",
                ""
            ),

            "seccion_origen": pagina.get(
                "seccion_origen",
                ""
            ),

            "contenido": contenido,

            "encabezados": encabezados

        }


        paginas_extraidas.append(
            pagina_extraida
        )


        print(
            f"Caracteres extraídos: "
            f"{len(contenido)}"
        )


        print(
            f"Encabezados: "
            f"{len(encabezados)}"
        )


    except Exception as e:

        print(
            "ERROR:",
            e
        )


# ==========================================================
# RESULTADO
# ==========================================================

print()
print("=" * 70)
print("EXTRACCIÓN COMPLETADA")
print("=" * 70)


if pagina_padre_extraida:

    print(
        "Página padre:",
        len(
            pagina_padre_extraida.get(
                "contenido",
                ""
            )
        ),
        "caracteres"
    )


print(
    "Páginas hijas previstas:",
    len(paginas_hijas)
)


print(
    "Páginas hijas extraídas correctamente:",
    len(paginas_extraidas)
)


print()


# ==========================================================
# MUESTRA DE LAS PÁGINAS EXTRAÍDAS
# ==========================================================

for pagina in paginas_extraidas:

    print("-" * 70)


    print(
        "Título:",
        pagina["titulo"]
    )


    print(
        "Tipo:",
        pagina["tipo"]
    )


    print(
        "Padre:",
        pagina["padre_origen"]
    )


    print(
        "Sección:",
        pagina["seccion_origen"]
    )


    print(
        "URL:",
        pagina["url"]
    )


    print(
        "Caracteres:",
        len(
            pagina["contenido"]
        )
    )


    print(
        "Primeros 500 caracteres:"
    )


    print(
        pagina["contenido"][:500]
    )


    print()


EXTRACCIÓN DE PÁGINA PADRE
Título: Iniciativas de I+D+i
Caracteres: 2260
Encabezados: 12

Páginas hijas obtenidas del JSON: 11

EXTRACCIÓN DE PÁGINAS HIJAS

[1/11] Calendario de convocatorias
URL: https://www.upv.es/pls/omag/ctt_agenda.Convocatorias_Cal
Caracteres extraídos: 2436
Encabezados: 4

[2/11] Buscador de convocatorias
URL: https://www.upv.es/pls/sogec/ctt_w05.Convocatorias_Bus?p_idioma=c
Caracteres extraídos: 1113
Encabezados: 2

[3/11] Ayudas nacionales y autonómicas
URL: https://www.upv.es/pls/somag/CTT_W04.Boletines?p_idioma=c&p_vista=compatible&p_grupo=1
Caracteres extraídos: 55413
Encabezados: 2

[4/11] Ayudas europeas
URL: https://www.upv.es/pls/somag/CTT_W04.Boletines?p_idioma=&p_vista=compatible&p_grupo=2
Caracteres extraídos: 21848
Encabezados: 2

[5/11] Contratación de investigadores
URL: https://www.upv.es/entidades/SRH/conypi/590724normalc.html
Caracteres extraídos: 8683
Encabezados: 0

[6/11] Programa de apoyo a la I+D
URL: https://www.upv.es/entidades/VINV/info

In [ ]:
# ==========================================================
# BLOQUE 11. Comprobación final antes de generar Markdown
#
# Comprueba:
# - página padre
# - páginas hijas
# - contenido vacío
# - contenido excesivamente corto
# - encabezados
# - posibles páginas con contenido duplicado
#
# Para la comprobación se utiliza el contenido que realmente
# será empleado posteriormente por el Bloque 13:
#
#     contenido_markdown
#
# Si no existe, se utiliza como alternativa:
#
#     contenido
#
# NO genera todavía archivos Markdown.
# ==========================================================


print()
print("=" * 70)
print("COMPROBACIÓN FINAL ANTES DE GENERAR MARKDOWN")
print("=" * 70)


# ==========================================================
# FUNCIÓN AUXILIAR PARA OBTENER EL CONTENIDO EFECTIVO
# ==========================================================

def obtener_contenido_efectivo(pagina):

    """
    Devuelve el contenido que se utilizará posteriormente
    para generar el Markdown.

    Se prioriza contenido_markdown y, si no existe,
    se utiliza contenido.
    """

    contenido = pagina.get(
        "contenido_markdown",
        ""
    )

    if not contenido:

        contenido = pagina.get(
            "contenido",
            ""
        )

    return contenido.strip()


# ==========================================================
# 1. COMPROBACIÓN DE LA PÁGINA PADRE
# ==========================================================

print()
print("-" * 70)
print("PÁGINA PADRE")
print("-" * 70)


if pagina_padre_extraida:

    titulo_padre = pagina_padre_extraida.get(
        "titulo",
        ""
    )

    url_padre = pagina_padre_extraida.get(
        "url",
        ""
    )

    tipo_padre = pagina_padre_extraida.get(
        "tipo",
        ""
    )

    contenido_padre = pagina_padre_extraida.get(
        "contenido",
        ""
    ).strip()

    encabezados_padre = pagina_padre_extraida.get(
        "encabezados",
        []
    )


    print(
        "Título:",
        titulo_padre
    )

    print(
        "URL:",
        url_padre
    )

    print(
        "Tipo:",
        tipo_padre
    )

    print(
        "Caracteres:",
        len(contenido_padre)
    )

    print(
        "Encabezados:",
        len(encabezados_padre)
    )


else:

    print(
        "ERROR: no existe la página padre."
    )


# ==========================================================
# 2. COMPROBACIÓN DE LAS PÁGINAS HIJAS
# ==========================================================

print()
print("-" * 70)
print("PÁGINAS HIJAS")
print("-" * 70)


print(
    "Páginas previstas:",
    len(paginas_hijas)
)

print(
    "Páginas extraídas:",
    len(paginas_extraidas)
)


# ==========================================================
# 3. DETECCIÓN DE PROBLEMAS
# ==========================================================

paginas_sin_contenido = []

paginas_contenido_corto = []

paginas_sin_encabezados = []


for pagina in paginas_extraidas:

    titulo = pagina.get(
        "titulo",
        ""
    )

    contenido = obtener_contenido_efectivo(
        pagina
    )

    encabezados = pagina.get(
        "encabezados",
        []
    )


    # ------------------------------------------------------
    # Sin contenido
    # ------------------------------------------------------

    if not contenido:

        paginas_sin_contenido.append(
            titulo
        )


    # ------------------------------------------------------
    # Contenido sospechosamente corto
    # ------------------------------------------------------

    elif len(contenido) < 500:

        paginas_contenido_corto.append({

            "titulo": titulo,

            "caracteres": len(contenido)

        })


    # ------------------------------------------------------
    # Sin encabezados
    #
    # No se considera automáticamente un error.
    # Algunas páginas pueden tener contenido válido sin una
    # estructura explícita de encabezados.
    # ------------------------------------------------------

    if not encabezados:

        paginas_sin_encabezados.append(
            titulo
        )


# ==========================================================
# 4. DETECCIÓN DE CONTENIDO DUPLICADO
#
# Se compara el contenido efectivo que utilizará el
# generador de Markdown.
#
# Esto permite detectar que varias páginas representan
# realmente el mismo recurso y evitar almacenarlo varias
# veces.
# ==========================================================

contenidos = {}


for pagina in paginas_extraidas:

    contenido = obtener_contenido_efectivo(
        pagina
    )

    if not contenido:

        continue


    if contenido not in contenidos:

        contenidos[contenido] = []


    contenidos[contenido].append(
        pagina.get(
            "titulo",
            ""
        )
    )


grupos_duplicados = []


for contenido, titulos in contenidos.items():

    if len(titulos) > 1:

        grupos_duplicados.append(
            titulos
        )


# ==========================================================
# 5. RESUMEN INDIVIDUAL
# ==========================================================

print()
print("-" * 70)
print("RESUMEN DE CADA RECURSO")
print("-" * 70)


for indice, pagina in enumerate(
    paginas_extraidas,
    start=1
):

    titulo = pagina.get(
        "titulo",
        ""
    )

    seccion = pagina.get(
        "seccion_origen",
        ""
    )

    tipo = pagina.get(
        "tipo",
        ""
    )

    url = pagina.get(
        "url",
        ""
    )

    contenido = obtener_contenido_efectivo(
        pagina
    )

    encabezados = pagina.get(
        "encabezados",
        []
    )


    print()

    print(
        f"[{indice}/{len(paginas_extraidas)}] "
        f"{titulo}"
    )

    print(
        f"  Sección     : {seccion}"
    )

    print(
        f"  Tipo        : {tipo}"
    )

    print(
        f"  Caracteres  : {len(contenido)}"
    )

    print(
        f"  Encabezados : {len(encabezados)}"
    )

    print(
        f"  URL         : {url}"
    )


# ==========================================================
# 6. RESULTADO DE LAS COMPROBACIONES
# ==========================================================

print()
print("=" * 70)
print("RESULTADO DE LA COMPROBACIÓN")
print("=" * 70)


# ----------------------------------------------------------
# Sin contenido
# ----------------------------------------------------------

if paginas_sin_contenido:

    print()

    print(
        "PÁGINAS SIN CONTENIDO:"
    )

    for titulo in paginas_sin_contenido:

        print(
            f"  - {titulo}"
        )

else:

    print()

    print(
        "OK: no hay páginas sin contenido."
    )


# ----------------------------------------------------------
# Contenido corto
# ----------------------------------------------------------

if paginas_contenido_corto:

    print()

    print(
        "PÁGINAS CON CONTENIDO CORTO:"
    )

    for pagina in paginas_contenido_corto:

        print(
            f"  - {pagina['titulo']} "
            f"({pagina['caracteres']} caracteres)"
        )

else:

    print()

    print(
        "OK: no hay páginas con contenido "
        "sospechosamente corto."
    )


# ----------------------------------------------------------
# Sin encabezados
#
# Se informa, pero no se considera por sí mismo un fallo.
# ----------------------------------------------------------

if paginas_sin_encabezados:

    print()

    print(
        "PÁGINAS SIN ENCABEZADOS:"
    )

    for titulo in paginas_sin_encabezados:

        print(
            f"  - {titulo}"
        )

else:

    print()

    print(
        "OK: todas las páginas tienen encabezados."
    )


# ----------------------------------------------------------
# Duplicados
# ----------------------------------------------------------

if grupos_duplicados:

    print()

    print(
        "CONTENIDOS DUPLICADOS DETECTADOS:"
    )

    for grupo in grupos_duplicados:

        print()

        for titulo in grupo:

            print(
                f"  - {titulo}"
            )

else:

    print()

    print(
        "OK: no se han detectado contenidos "
        "completamente duplicados."
    )


# ==========================================================
# 7. INFORMACIÓN SOBRE CONTENIDO SEMÁNTICO
#
# Permite comprobar si las páginas ya disponen de una
# representación específica para Markdown.
# ==========================================================

paginas_con_contenido_markdown = 0

paginas_sin_contenido_markdown = 0


for pagina in paginas_extraidas:

    contenido_markdown = pagina.get(
        "contenido_markdown",
        ""
    ).strip()


    if contenido_markdown:

        paginas_con_contenido_markdown += 1

    else:

        paginas_sin_contenido_markdown += 1


print()
print("-" * 70)
print("REPRESENTACIÓN SEMÁNTICA")
print("-" * 70)

print(
    "Páginas con contenido_markdown:",
    paginas_con_contenido_markdown
)

print(
    "Páginas sin contenido_markdown:",
    paginas_sin_contenido_markdown
)


# ==========================================================
# 8. DECISIÓN FINAL
# ==========================================================
#
# La ausencia de encabezados NO bloquea la generación,
# porque existen páginas válidas que pueden no tenerlos.
#
# Los problemas que sí bloquean son:
# - páginas sin contenido
# - contenido sospechosamente corto
#
# Los duplicados se muestran para que el Bloque 13 pueda
# agruparlos y almacenarlos una sola vez.
# ==========================================================

hay_problemas = (

    len(paginas_sin_contenido) > 0

    or

    len(paginas_contenido_corto) > 0

)


print()
print("=" * 70)


if hay_problemas:

    print(
        "REVISIÓN NECESARIA ANTES DE GENERAR MARKDOWN"
    )

    print(
        "Se han detectado posibles problemas "
        "en la extracción."
    )

else:

    print(
        "EXTRACCIÓN LISTA PARA GENERAR MARKDOWN"
    )

    print(
        "Los contenidos duplicados serán agrupados "
        "por el Bloque 13."
    )


print("=" * 70)


COMPROBACIÓN FINAL ANTES DE GENERAR MARKDOWN

----------------------------------------------------------------------
PÁGINA PADRE
----------------------------------------------------------------------
Título: Iniciativas de I+D+i
URL: https://www.upv.es/investigacion/iniciativas-idi/index-es.html
Tipo: padre
Caracteres: 2260
Encabezados: 12

----------------------------------------------------------------------
PÁGINAS HIJAS
----------------------------------------------------------------------
Páginas previstas: 11
Páginas extraídas: 11

----------------------------------------------------------------------
RESUMEN DE CADA RECURSO
----------------------------------------------------------------------

[1/11] Calendario de convocatorias
  Sección     : Convocatorias públicas
  Tipo        : dinamico
  Caracteres  : 1254
  Encabezados : 4
  URL         : https://www.upv.es/pls/omag/ctt_agenda.Convocatorias_Cal

[2/11] Buscador de convocatorias
  Sección     : Convocatorias públicas
  T

In [ ]:
# ==========================================================
# BLOQUE 12. PREPARACIÓN DE CONTENIDO MARKDOWN
#
# Tratamiento específico de páginas dinámicas.
#
# Las páginas dinámicas de la UPV suelen contener:
# - títulos de convocatorias
# - listas de elementos
# - párrafos informativos
#
# Se conserva una estructura Markdown básica para facilitar
# posteriormente la recuperación mediante RAG.
#
# Las páginas normales NO se modifican.
# ==========================================================

from bs4 import BeautifulSoup
import re


# ==========================================================
# FUNCIÓN PARA EXTRAER DINÁMICAS COMO MARKDOWN
# ==========================================================

def extraer_dinamico_markdown(html):

    soup = BeautifulSoup(
        html,
        "html.parser"
    )

    # ------------------------------------------------------
    # Eliminar elementos que no aportan contenido
    # ------------------------------------------------------

    for elemento in soup([
        "script",
        "style",
        "noscript",
        "nav",
        "header",
        "footer"
    ]):

        elemento.decompose()


    # ------------------------------------------------------
    # Buscar contenedor principal
    # ------------------------------------------------------

    selectores = [
        "main",
        "article",
        ".entry-content",
        "#content",
        ".content",
        ".entry"
    ]

    contenedor = None

    for selector in selectores:

        elemento = soup.select_one(
            selector
        )

        if elemento:

            texto = elemento.get_text(
                " ",
                strip=True
            )

            if len(texto) > 200:

                contenedor = elemento

                break


    # ------------------------------------------------------
    # Si no se encuentra contenedor,
    # utilizar body
    # ------------------------------------------------------

    if contenedor is None:

        contenedor = soup.body

    if contenedor is None:

        contenedor = soup


    # ------------------------------------------------------
    # Construir Markdown
    # ------------------------------------------------------

    bloques = []


    for elemento in contenedor.find_all(
        [
            "h1",
            "h2",
            "h3",
            "h4",
            "p",
            "li"
        ]
    ):

        texto = elemento.get_text(
            " ",
            strip=True
        )

        if not texto:

            continue


        texto = limpiar_texto(
            texto
        )


        # --------------------------------------------------
        # Encabezados
        # --------------------------------------------------

        if elemento.name == "h1":

            bloques.append(
                f"# {texto}"
            )

        elif elemento.name == "h2":

            bloques.append(
                f"## {texto}"
            )

        elif elemento.name == "h3":

            bloques.append(
                f"### {texto}"
            )

        elif elemento.name == "h4":

            bloques.append(
                f"#### {texto}"
            )


        # --------------------------------------------------
        # Listas
        # --------------------------------------------------

        elif elemento.name == "li":

            bloques.append(
                f"- {texto}"
            )


        # --------------------------------------------------
        # Párrafos
        # --------------------------------------------------

        elif elemento.name == "p":

            bloques.append(
                texto
            )


    # ------------------------------------------------------
    # Unir bloques
    # ------------------------------------------------------

    contenido = "\n\n".join(
        bloques
    )


    # ------------------------------------------------------
    # Limpieza de saltos excesivos
    # ------------------------------------------------------

    contenido = re.sub(
        r"\n{3,}",
        "\n\n",
        contenido
    )


    return contenido.strip()


# ==========================================================
# GENERAR CONTENIDO MARKDOWN PARA LAS PÁGINAS DINÁMICAS
# ==========================================================

print()
print("=" * 70)
print("PREPARACIÓN DE PÁGINAS DINÁMICAS")
print("=" * 70)


paginas_extraidas_markdown = []


for indice, pagina in enumerate(
    paginas_extraidas,
    start=1
):

    # ------------------------------------------------------
    # Las páginas normales mantienen su contenido original
    # ------------------------------------------------------

    if pagina.get("tipo") != "dinamico":

        pagina["contenido_markdown"] = pagina.get(
            "contenido",
            ""
        )

        paginas_extraidas_markdown.append(
            pagina
        )

        continue


    print()
    print(
        f"[{indice}/{len(paginas_extraidas)}] "
        f"{pagina.get('titulo', '')}"
    )

    try:

        # --------------------------------------------------
        # Volver a descargar HTML para conservar estructura
        # --------------------------------------------------

        html = descargar_html(
            pagina["url"]
        )


        # --------------------------------------------------
        # Extraer como Markdown
        # --------------------------------------------------

        contenido_markdown = extraer_dinamico_markdown(
            html
        )


        pagina["contenido_markdown"] = (
            contenido_markdown
        )


        paginas_extraidas_markdown.append(
            pagina
        )


        print(
            "Contenido original:",
            len(
                pagina.get(
                    "contenido",
                    ""
                )
            ),
            "caracteres"
        )

        print(
            "Contenido Markdown:",
            len(
                contenido_markdown
            ),
            "caracteres"
        )


    except Exception as e:

        print(
            "ERROR:",
            e
        )

        # --------------------------------------------------
        # Si falla, conservar extracción original
        # --------------------------------------------------

        pagina["contenido_markdown"] = pagina.get(
            "contenido",
            ""
        )

        paginas_extraidas_markdown.append(
            pagina
        )


# ==========================================================
# RESULTADO
# ==========================================================

print()
print("=" * 70)
print("PREPARACIÓN COMPLETADA")
print("=" * 70)

print(
    "Páginas procesadas:",
    len(paginas_extraidas_markdown)
)

print(
    "Páginas dinámicas:",
    sum(
        1
        for p in paginas_extraidas_markdown
        if p.get("tipo") == "dinamico"
    )
)

print(
    "Páginas normales:",
    sum(
        1
        for p in paginas_extraidas_markdown
        if p.get("tipo") != "dinamico"
    )
)


PREPARACIÓN DE PÁGINAS DINÁMICAS

[1/11] Calendario de convocatorias
Contenido original: 2436 caracteres
Contenido Markdown: 1254 caracteres

[2/11] Buscador de convocatorias
Contenido original: 1113 caracteres
Contenido Markdown: 1169 caracteres

[3/11] Ayudas nacionales y autonómicas
Contenido original: 55413 caracteres
Contenido Markdown: 55813 caracteres

[4/11] Ayudas europeas
Contenido original: 21848 caracteres
Contenido Markdown: 21783 caracteres

PREPARACIÓN COMPLETADA
Páginas procesadas: 11
Páginas dinámicas: 4
Páginas normales: 7


In [ ]:
# ==========================================================
# BLOQUE 13. GENERACIÓN DE FICHEROS MARKDOWN
#
# Estructura:
#
# INVESTIGACION/
# └── iniciativas_idi/
#     ├── iniciativas_idi.md
#     └── recursos/
#         ├── calendario_convocatorias.md
#         ├── buscador_convocatorias.md
#         ├── ayudas_nacionales_autonomicas.md
#         ├── ayudas_europeas.md
#         ├── contratacion_investigadores.md
#         └── comite_etica_investigacion.md
#
# Las páginas con contenido idéntico se almacenan una sola vez.
#
# El Markdown intenta representar la estructura semántica
# de las páginas, evitando reproducir simplemente el HTML.
# ==========================================================

import re
import hashlib


# ==========================================================
# 1. FUNCIÓN PARA GENERAR NOMBRES DE FICHERO
# ==========================================================

def nombre_archivo_markdown(titulo):

    nombre = titulo.lower()

    # Sustituir caracteres acentuados
    reemplazos = {
        "á": "a",
        "é": "e",
        "í": "i",
        "ó": "o",
        "ú": "u",
        "ü": "u",
        "ñ": "n"
    }

    for origen, destino in reemplazos.items():

        nombre = nombre.replace(
            origen,
            destino
        )

    # Sustituir caracteres no alfanuméricos
    nombre = re.sub(
        r"[^a-z0-9]+",
        "_",
        nombre
    )

    nombre = nombre.strip("_")

    return nombre + ".md"


# ==========================================================
# 2. FUNCIÓN PARA NORMALIZAR CONTENIDO
# ==========================================================

def hash_contenido(contenido):

    return hashlib.sha256(
        contenido.strip().encode(
            "utf-8"
        )
    ).hexdigest()


# ==========================================================
# 3. FUNCIÓN PARA CONSTRUIR ESTRUCTURA SEMÁNTICA
# ==========================================================

def generar_estructura_semantica(
    contenido,
    encabezados
):
    """
    Construye una representación Markdown más semántica
    a partir del contenido extraído y de los encabezados.

    Los encabezados HTML se utilizan como estructura
    jerárquica y no se muestran como información técnica
    del tipo 'h1', 'h2', 'h3'.

    Si no existen encabezados útiles, se conserva el
    contenido textual original.
    """

    if not encabezados:

        return contenido.strip()


    # ------------------------------------------------------
    # Normalizar encabezados
    # ------------------------------------------------------

    encabezados_validos = []

    for encabezado in encabezados:

        texto = encabezado.get(
            "texto",
            ""
        ).strip()

        nivel = encabezado.get(
            "nivel",
            ""
        )

        if not texto:
            continue

        if nivel not in [
            "h1",
            "h2",
            "h3"
        ]:
            continue

        encabezados_validos.append({
            "nivel": nivel,
            "texto": texto
        })


    if not encabezados_validos:

        return contenido.strip()


    # ------------------------------------------------------
    # Crear Markdown a partir de la jerarquía detectada
    #
    # h1 -> #
    # h2 -> ##
    # h3 -> ###
    # ------------------------------------------------------

    estructura = []

    for encabezado in encabezados_validos:

        nivel_html = encabezado["nivel"]

        nivel_markdown = {
            "h1": 1,
            "h2": 2,
            "h3": 3
        }[nivel_html]

        estructura.append(
            "#" * nivel_markdown
            + " "
            + encabezado["texto"]
        )


    # ------------------------------------------------------
    # Añadir el contenido textual como contexto general
    #
    # Se conserva porque la extracción actual no mantiene
    # la relación exacta entre cada párrafo y su encabezado.
    # ------------------------------------------------------

    estructura.append("")

    estructura.append(
        contenido.strip()
    )


    return "\n\n".join(
        estructura
    )


# ==========================================================
# 4. GENERAR MARKDOWN DE LA PÁGINA PADRE
# ==========================================================

contenido_padre = pagina_padre_extraida.get(
    "contenido",
    ""
).strip()

encabezados_padre = pagina_padre_extraida.get(
    "encabezados",
    []
)

estructura_padre = generar_estructura_semantica(
    contenido_padre,
    encabezados_padre
)


with open(
    RUTA_MARKDOWN_PADRE,
    "w",
    encoding="utf-8"
) as archivo:

    # ------------------------------------------------------
    # Título principal
    # ------------------------------------------------------

    archivo.write(
        f"# {pagina_padre_extraida['titulo']}\n\n"
    )


    # ------------------------------------------------------
    # Metadatos
    # ------------------------------------------------------

    archivo.write(
        f"**Tipo:** {pagina_padre_extraida['tipo']}\n\n"
    )

    archivo.write(
        f"**URL:** {pagina_padre_extraida['url']}\n\n"
    )


    # ------------------------------------------------------
    # Contenido semántico
    # ------------------------------------------------------

    archivo.write(
        estructura_padre
    )

    archivo.write(
        "\n"
    )


# ==========================================================
# 5. AGRUPAR PÁGINAS HIJAS POR CONTENIDO
# ==========================================================

grupos_contenido = {}

for pagina in paginas_extraidas:

    contenido = pagina.get(
        "contenido_markdown",
        pagina.get(
            "contenido",
            ""
        )
    ).strip()


    if not contenido:

        continue


    clave = hash_contenido(
        contenido
    )


    if clave not in grupos_contenido:

        grupos_contenido[clave] = []


    grupos_contenido[clave].append(
        pagina
    )


# ==========================================================
# 6. GENERAR MARKDOWN DE LOS RECURSOS
# ==========================================================

recursos_generados = 0

print()
print("=" * 70)
print("GENERACIÓN DE MARKDOWN")
print("=" * 70)


for grupo in grupos_contenido.values():

    # ------------------------------------------------------
    # Página representativa del grupo
    # ------------------------------------------------------

    pagina_principal = grupo[0]


    nombre_archivo = nombre_archivo_markdown(
        pagina_principal["titulo"]
    )


    ruta_archivo = os.path.join(
        CARPETA_RECURSOS,
        nombre_archivo
    )


    # ------------------------------------------------------
    # Contenido
    # ------------------------------------------------------

    contenido = pagina_principal.get(
        "contenido_markdown",
        pagina_principal.get(
            "contenido",
            ""
        )
    ).strip()


    encabezados = pagina_principal.get(
        "encabezados",
        []
    )


    # ------------------------------------------------------
    # Construir estructura semántica
    # ------------------------------------------------------

    estructura_semantica = generar_estructura_semantica(
        contenido,
        encabezados
    )


    # ------------------------------------------------------
    # Generar Markdown
    # ------------------------------------------------------

    with open(
        ruta_archivo,
        "w",
        encoding="utf-8"
    ) as archivo:

        # --------------------------------------------------
        # Título
        # --------------------------------------------------

        archivo.write(
            f"# {pagina_principal['titulo']}\n\n"
        )


        # --------------------------------------------------
        # Metadatos
        # --------------------------------------------------

        archivo.write(
            f"**Tipo:** "
            f"{pagina_principal['tipo']}\n\n"
        )

        archivo.write(
            f"**Página padre:** "
            f"{pagina_principal['padre_origen']}\n\n"
        )

        archivo.write(
            f"**Sección:** "
            f"{pagina_principal['seccion_origen']}\n\n"
        )


        # --------------------------------------------------
        # URLs asociadas
        #
        # Si varias páginas comparten contenido, se conservan
        # todas las URLs como referencias al mismo recurso.
        # --------------------------------------------------

        archivo.write(
            "## URLs asociadas\n\n"
        )


        for pagina in grupo:

            archivo.write(
                f"- {pagina['url']}\n"
            )


        # --------------------------------------------------
        # Contenido semántico
        # --------------------------------------------------

        archivo.write(
            "\n"
        )

        archivo.write(
            estructura_semantica
        )

        archivo.write(
            "\n"
        )


    recursos_generados += 1


    print(
        f"[{recursos_generados}/"
        f"{len(grupos_contenido)}] "
        f"{nombre_archivo}"
    )


    if len(grupo) > 1:

        print(
            f"    Contenido compartido por "
            f"{len(grupo)} páginas"
        )


# ==========================================================
# 7. RESUMEN FINAL
# ==========================================================

print()
print("=" * 70)
print("GENERACIÓN COMPLETADA")
print("=" * 70)

print()

print(
    "Directorio principal:"
)

print(
    CARPETA_INICIATIVAS_IDI
)

print()

print(
    "Markdown de página padre:"
)

print(
    RUTA_MARKDOWN_PADRE
)

print()

print(
    "Páginas hijas extraídas:",
    len(paginas_extraidas)
)

print(
    "Grupos de contenido únicos:",
    len(grupos_contenido)
)

print(
    "Recursos Markdown generados:",
    recursos_generados
)

print()

print(
    "Estructura generada:"
)

print(
    "INVESTIGACION/"
)

print(
    "└── iniciativas_idi/"
)

print(
    "    ├── iniciativas_idi.md"
)

print(
    "    └── recursos/"
)


for grupo in grupos_contenido.values():

    nombre = nombre_archivo_markdown(
        grupo[0]["titulo"]
    )

    print(
        f"        ├── {nombre}"
    )


GENERACIÓN DE MARKDOWN
[1/7] calendario_de_convocatorias.md
[2/7] buscador_de_convocatorias.md
[3/7] ayudas_nacionales_y_autonomicas.md
[4/7] ayudas_europeas.md
[5/7] contratacion_de_investigadores.md
[6/7] programa_de_apoyo_a_la_i_d.md
    Contenido compartido por 5 páginas
[7/7] comite_de_etica_en_la_investigacion.md

GENERACIÓN COMPLETADA

Directorio principal:
/content/drive/MyDrive/TFG Teleco/INVESTIGACION/iniciativas_idi

Markdown de página padre:
/content/drive/MyDrive/TFG Teleco/INVESTIGACION/iniciativas_idi/iniciativas_idi.md

Páginas hijas extraídas: 11
Grupos de contenido únicos: 7
Recursos Markdown generados: 7

Estructura generada:
INVESTIGACION/
└── iniciativas_idi/
    ├── iniciativas_idi.md
    └── recursos/
        ├── calendario_de_convocatorias.md
        ├── buscador_de_convocatorias.md
        ├── ayudas_nacionales_y_autonomicas.md
        ├── ayudas_europeas.md
        ├── contratacion_de_investigadores.md
        ├── programa_de_apoyo_a_la_i_d.md
        ├── com